# Baseline Evaluation: Qwen 2.5 7B (Instruct & Coder)

Evaluates the baseline performance of Qwen 2.5 7B models (Instruct and Coder) in 4-bit quantization on the Trip Planning dataset.
Uses **5-shot prompting** with **forced prefix generation** to ensure consistent formatting.

**Style**: Matched to `evaluate_sft_phase.ipynb` using Unsloth.

In [ ]:
%%capture
# Install Unsloth and compatible libraries
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes wandb ortools

In [1]:
# --- 1. Imports ---
import torch
from unsloth import FastLanguageModel
import json
import os
import gc
from tqdm import tqdm

# --- CONFIG ---
max_seq_length = 8192
dtype = None # Auto detection
load_in_4bit = True

# Forced prefix to ensure the model follows the plan format immediately
forced_prefix = "Here is the trip plan for visiting the"

# Strict System Prompt (Refined based on user input)
system_instruction = (
    "You are a precise trip planning engine.\n"
    "TASK: Generate a valid travel itinerary that strictly satisfies all duration, sequence, and flight connectivity constraints.\n\n"
    "OUTPUT FORMAT (STRICT):\n"
    "- Use the format: '**Day X-Y:** Visit [City] for [N] days.' and '**Day X:** Fly from [City A] to [City B].'\n\n"
    "STRICT RULES:\n"
    "1. Output ONLY the final trip plan.\n"
    "2. NO explanations, NO reasoning traces, NO summaries, and NO conversational filler.\n"
    "3. DO NOT restate the task or constraints.\n"
    "4. Ensure the plan uses ONLY the direct flights listed.\n"
    "5. Keep the output concise and follow the example style EXACTLY.\n"
)

print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


🦥 Unsloth Zoo will now patch everything to make training faster!
Device: NVIDIA A40


In [2]:
# --- 2. Helper Functions ---

def evaluate_baseline_model(model_name, dataset, output_file, tag="Model", batch_size=8):
    """
    Loads a model using Unsloth, generates responses for the dataset in batches,
    and saves the results to output_file.
    """
    print(f"\nLoading {tag} from {model_name}...")
    
    # 1. Load Model (Clean State)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model)
    
    # REQUIRED for Batched Generation
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Load existing results if resuming
    results = {}
    if os.path.exists(output_file):
        try:
            with open(output_file, 'r') as f:
                results = json.load(f)
            print(f"Resuming from {len(results)} existing results.")
        except:
            print("Could not load existing file. Starting fresh.")
    
    # Filter keys to process
    keys_to_process = [k for k in dataset.keys() if k not in results]
    print(f"Remaining examples: {len(keys_to_process)}")
    
    # Process in Batches
    # We sort keys to potentially group similar lengths, though shuffling is also fine.
    # For reproducibility, we keep list order.
    pass
    
    save_frequency_batches = 2  # Save every 2 batches
    
    # Batching Loop
    for i in tqdm(range(0, len(keys_to_process), batch_size), desc=f"Eval {tag} (Batched)"):
        batch_keys = keys_to_process[i : i + batch_size]
        
        # Prepare Inputs
        batch_prompts = []
        valid_keys = []
        
        for key in batch_keys:
            example = dataset[key]
            user_prompt = example.get('prompt_5shot')
            if not user_prompt:
                print(f"\nWarning: No prompt_5shot for {key}")
                continue
            
            # Templating
            messages = [
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": user_prompt}
            ]
            prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            final_input_text = prompt_str + forced_prefix
            
            batch_prompts.append(final_input_text)
            valid_keys.append(key)
        
        if not batch_prompts:
            continue
            
        # Tokenize Batch
        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=False).to("cuda")
        input_len = inputs.input_ids.shape[1]
        
        # Generate Batch
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=4096, 
                use_cache=True, 
                do_sample=False, 
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            
        # Decode Batch
        generated_texts = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)
        
        # Save Results
        for key, gen_text in zip(valid_keys, generated_texts):
            full_response = forced_prefix + gen_text
            
            result_entry = dataset[key].copy()
            model_key_suffix = model_name.split('/')[-1].lower().replace('-', '_').replace('.', '_')
            result_entry[f'pred_5shot_{model_key_suffix}'] = full_response
            results[key] = result_entry
            
        # Periodic Save
        if (i // batch_size) % save_frequency_batches == 0:
            with open(output_file, 'w') as f:
                json.dump(results, f, indent=4)

    # Final Save
    with open(output_file, 'w') as f:
        json.dump(results, f, indent=4)
    
    print(f"\nSaved final results to {output_file}")
    
    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()
    print("VRAM Cleaned.")

In [3]:
# --- 3. Load Data ---
input_path = 'data/trip_planning_val.json'

print(f"Loading data from {input_path}...")
with open(input_path, 'r') as f:
    data = json.load(f)

print(f"Loaded {len(data)} examples.")

Loading data from data/trip_planning_val.json...
Loaded 160 examples.


In [4]:
# --- 4. Evaluate Models ---

models_to_eval = [
    {
        "name": "Qwen 2.5 7B Instruct",
        "path": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit", 
        # Folder: data/qwen2.5-7b, File: trip_planning_val_results_qwen2.5-7b.json
        "output": "data/qwen2.5-7b/trip_planning_val_results_qwen2.5-7b.json"
    },
    {
        "name": "Qwen 2.5 7B Coder",
        "path": "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
        # Folder: data/qwen2.5-7b-coder, File: trip_planning_val_results_qwen2.5-7b-coder.json
        "output": "data/qwen2.5-7b-coder/trip_planning_val_results_qwen2.5-7b-coder.json"
    }
]

for model_info in models_to_eval:
    # Ensure output dir exists
    output_dir = os.path.dirname(model_info['output'])
    if output_dir: # Check if not empty
        os.makedirs(output_dir, exist_ok=True)
        print(f"\nCreated/Checked directory: {output_dir}")
    
    evaluate_baseline_model(
        model_name=model_info['path'],
        dataset=data,
        output_file=model_info['output'],
        tag=model_info.get('name', 'Model')
    )


Created/Checked directory: data/qwen2.5-7b

Loading Qwen 2.5 7B Instruct from unsloth/Qwen2.5-7B-Instruct-bnb-4bit...
==((====))==  Unsloth 2026.1.2: Fast Qwen2 patching. Transformers: 4.57.1. vLLM: 0.11.2.
   \\   /|    NVIDIA A40. Num GPUs = 1. Max memory: 44.432 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Remaining examples: 160


Eval Qwen 2.5 7B Instruct (Batched): 100%|██████████| 20/20 [10:00<00:00, 30.04s/it]



Saved final results to data/qwen2.5-7b/trip_planning_val_results_qwen2.5-7b.json
VRAM Cleaned.

Created/Checked directory: data/qwen2.5-7b-coder

Loading Qwen 2.5 7B Coder from unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit...
==((====))==  Unsloth 2026.1.2: Fast Qwen2 patching. Transformers: 4.57.1. vLLM: 0.11.2.
   \\   /|    NVIDIA A40. Num GPUs = 1. Max memory: 44.432 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Remaining examples: 160


Eval Qwen 2.5 7B Coder (Batched): 100%|██████████| 20/20 [05:41<00:00, 17.08s/it]



Saved final results to data/qwen2.5-7b-coder/trip_planning_val_results_qwen2.5-7b-coder.json
VRAM Cleaned.
